<a href="https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 — CTR vs Position**

**Verdict: CONFIRMED**

Observed CTR decreases across the position buckets in the measured data: mean CTR is highest at positions 1–3 (2.7143%) and declines to 0.2113% at position 21+. The median CTR is less consistent because the distribution is sparse and skewed, so this supports a directional relationship rather than a causal claim.


**Signal 2 — Impressions / Exposure**

**Verdict: MIXED**

Higher-impression buckets show higher median CTR overall, rising from 0.00% in the lowest two buckets to 0.23% in the highest bucket. However, the pattern is not a clean monotonic relationship in the mean because the lowest-volume bucket contains strong outliers. I therefore treat impressions as an exposure/visibility condition rather than a standalone opportunity signal.


---
### My baseline rule

Review content when it has meaningful search exposure (`impressions_90d >= 3616`) but its average position is worse than page 1 (`avg_position > 10`).

The rule prioritizes items using a transparent score that increases with both search exposure and the size of the position gap beyond page 1. This is decision-support for review, not evidence that an item definitely needs a specific fix.

**Reason code:** `visible_but_page2plus`

**Action:** `REVIEW`

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/Sgln24/flyrank-ml-internship-w1.git /content/flyrank-ml-internship
%cd /content/flyrank-ml-internship

import os
import pandas as pd

print("Current directory:", os.getcwd())
print("\nTop-level files:")
print(os.listdir("."))

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print(df[["ctr", "avg_position"]].describe())
print("Rows with no position data:", (df["avg_position"] == 0).sum())
print("Rows with CTR missing:", df["ctr"].isna().sum())

position_check = df[df["avg_position"] > 0].copy()

position_check["position_bucket"] = pd.cut(
    position_check["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_table = (
    position_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(position_table)

print(df["search_volume"].describe())
print("\nMissing search_volume:", df["search_volume"].isna().sum())

print(
    df["search_volume"]
    .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0])
)

volume_check = df.copy()

volume_check["volume_bucket"] = pd.cut(
    volume_check["search_volume"],
    bins=[-1, 0, 20, 110, 390, float("inf")],
    labels=["0", "1-20", "21-110", "111-390", "391+"],
    include_lowest=True
)

volume_check["volume_bucket"] = volume_check["volume_bucket"].astype("object")
volume_check.loc[
    volume_check["search_volume"].isna(),
    "volume_bucket"
] = "missing"

volume_table = (
    volume_check
    .groupby("volume_bucket", dropna=False, observed=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(volume_table)

print(df["impressions_90d"].describe())
print(
    df["impressions_90d"]
    .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0])
)

print("Missing impressions_90d:", df["impressions_90d"].isna().sum())

impression_check = df.copy()

impression_check["impression_bucket"] = pd.cut(
    impression_check["impressions_90d"],
    bins=[0, 81, 731, 3615.25, 12136.4, float("inf")],
    labels=["1-81", "82-731", "732-3,615", "3,616-12,136", "12,137+"],
    include_lowest=True
)

impression_table = (
    impression_check
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        mean_impressions=("impressions_90d", "mean")
    )
    .reset_index()
)

display(impression_table)

rule_candidate = (
    (df["impressions_90d"] >= 3616)
    & (df["avg_position"] > 10)
)

print("Candidate rows:", rule_candidate.sum())
print("Candidate rate:", rule_candidate.mean())

display(
    df.loc[
        rule_candidate,
        ["content_id", "impressions_90d", "avg_position", "ctr"]
    ].describe()
)

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship
Current directory: /content/flyrank-ml-internship

Top-level files:
['submission', 'DATA_USE.md', 'notebooks', '.github', 'outputs', 'GUIDE.md', 'requirements.txt', 'CLAUDE.md', 'docs', 'scripts', 'README.md', 'skills', 'data', 'SETUP.md', 'AGENTS.md', '.gitignore', 'LICENSE', 'work', '.git']
Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order

,position_bucket,n,mean_ctr,median_ctr
0,1-3,1141,2.714303,0.00
1,4-10,11842,0.651045,0.16
2,11-20,7273,0.323443,0.10
3,21+,8539,0.211333,0.00


count    27532.000000
mean       158.882391
std       1518.270825
min          0.000000
25%          0.000000
50%         10.000000
75%         20.000000
max      74000.000000
Name: search_volume, dtype: float64

Missing search_volume: 2468
0.00        0.0
0.25        0.0
0.50       10.0
0.75       20.0
0.90      110.0
0.95      390.0
1.00    74000.0
Name: search_volume, dtype: float64


,volume_bucket,n,mean_ctr,median_ctr
0,0,11081,0.437774,0.09
1,1-20,9601,0.275737,0.10
2,111-390,1476,0.196416,0.03
3,21-110,4160,0.207986,0.07
4,391+,1214,0.150832,0.00
5,missing,2468,2.627812,0.00


count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64
0.00         1.00
0.25        81.00
0.50       731.00
0.75      3615.25
0.90     12136.40
0.95     22996.50
1.00    517715.00
Name: impressions_90d, dtype: float64
Missing impressions_90d: 0


,impression_bucket,n,mean_ctr,median_ctr,mean_impressions
0,1-81,7503,1.265650,0.00,19.668533
1,82-731,7499,0.237681,0.00,334.827310
2,"732-3,615",7498,0.228640,0.13,1785.258069
3,"3,616-12,136",4500,0.303124,0.21,6765.902667
4,"12,137+",3000,0.321687,0.23,36506.706333


Candidate rows: 3285
Candidate rate: 0.1095


,impressions_90d,avg_position,ctr
count,3285.000000,3285.000000,3285.000000
mean,14983.961948,23.058661,0.252670
std,22787.833882,10.314383,0.285626
min,3616.000000,10.100000,0.000000
25%,5270.000000,14.500000,0.070000
50%,8342.000000,21.500000,0.160000
75%,15826.000000,29.500000,0.330000
max,497727.000000,85.800000,2.620000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring and ranked queue

The baseline assigns a transparent review score to content with meaningful search exposure (`impressions_90d >= 3616`) and an average position worse than page 1 (`avg_position > 10`).

The score is calculated as:

`impressions_90d × (avg_position − 10)`

This gives higher priority to items with greater search exposure and a larger position gap beyond page 1. Each item receives one reason code and one action label, then the full dataset is ranked by score.

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

The CSV is regenerated by the notebook on each run.

In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import os

baseline = df.copy()

# The rule:
# meaningful exposure + position worse than page 1
qualifies = (
    (baseline["impressions_90d"] >= 3616)
    & (baseline["avg_position"] > 10)
)

# Transparent, unfitted score:
# more impressions + larger position gap = higher review priority
baseline["score"] = np.where(
    qualifies,
    baseline["impressions_90d"] * (baseline["avg_position"] - 10),
    0
)

# One reason code and one action label
baseline["reason_code"] = np.where(
    qualifies,
    "visible_but_page2plus",
    "no_action"
)

baseline["action"] = np.where(
    qualifies,
    "REVIEW",
    "NO_ACTION"
)

# Rank highest-priority items first
baseline = baseline.sort_values(
    ["score", "impressions_90d", "avg_position"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Keep the queue readable
queue = baseline[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
].copy()

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Wrote: {output_path}")
print(f"Rows: {len(queue):,}")
print(f"Review rows: {(queue['action'] == 'REVIEW').sum():,}")
print(f"Non-review rows: {(queue['action'] == 'NO_ACTION').sum():,}")

display(queue.head(20))

print("Top score:", queue["score"].iloc[0])
print("Minimum positive score:", queue.loc[queue["score"] > 0, "score"].min())

Wrote: work/outputs/baseline_action_score.csv
Rows: 30,000
Review rows: 3,285
Non-review rows: 26,715


,rank,content_id,score,reason_code,action,impressions_90d,avg_position,ctr
0,1,content_a023517539fe,16224762.6,visible_but_page2plus,REVIEW,214047,85.8,0.01
1,2,content_2dba2b1f9536,7937468.6,visible_but_page2plus,REVIEW,443434,27.9,0.21
2,3,content_2cb567c3c89b,6072269.4,visible_but_page2plus,REVIEW,497727,22.2,0.10
3,4,content_54baba704595,4832829.0,visible_but_page2plus,REVIEW,130617,47.0,0.01
4,5,content_b28d1efd668f,4643049.6,visible_but_page2plus,REVIEW,286608,26.2,0.06
5,6,content_109f8f7c9d39,4017134.4,visible_but_page2plus,REVIEW,90476,54.4,0.01
6,7,content_ff94c9b6b411,3977048.4,visible_but_page2plus,REVIEW,228566,27.4,0.04
7,8,content_88d367c507a3,3941053.2,visible_but_page2plus,REVIEW,130932,40.1,0.04
8,9,content_813e88069237,3783688.2,visible_but_page2plus,REVIEW,233561,26.2,0.06
9,10,content_b511d4bc4ad2,3685878.5,visible_but_page2plus,REVIEW,205915,27.9,0.14


Top score: 16224762.6
Minimum positive score: 381.89999999999867


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 manual review

All ten items are marked `REVIEW` with the reason code `visible_but_page2plus`. They qualify because they have meaningful search exposure while ranking worse than page 1. The score then prioritizes items with greater exposure and a larger position gap.

| Rank | Action | Why it's here | Confidence note | What would make it wrong |
|---|---|---|---|---|
| 1 | REVIEW | 214,047 impressions with position 85.8 creates a very large exposure-position priority signal. | Medium: the signal is strong, but the rule is intentionally simple. | The high impression count could come from a broad query mix or an item whose position is not actionable through a content change. |
| 2 | REVIEW | 443,434 impressions with position 27.9 gives high exposure combined with a substantial page-1 gap. | Medium. | The observed position may reflect query mix, SERP characteristics, or intent rather than a fixable content issue. |
| 3 | REVIEW | 497,727 impressions with position 22.2 gives the largest exposure among the reviewed items while remaining outside page 1. | Medium. | Very high impressions may reflect many low-value queries, so the apparent opportunity may not translate into a useful action. |
| 4 | REVIEW | 130,617 impressions with position 47.0 produces a large exposure-position score. | Medium. | The page may be difficult to improve through content changes, or the ranking may be driven by factors outside the content itself. |
| 5 | REVIEW | 286,608 impressions with position 26.2 combines substantial exposure with a clear page-1 gap. | Medium. | High impressions do not establish that improving the page would increase clicks or rankings. |
| 6 | REVIEW | 90,476 impressions with position 54.4 creates a high score because the page is far beyond page 1 despite meaningful exposure. | Medium. | The position may reflect highly competitive queries or intent mismatch that this rule cannot diagnose. |
| 7 | REVIEW | 228,566 impressions with position 27.4 gives both strong exposure and a substantial position gap. | Medium. | The opportunity could be overstated if impressions come from queries that are not strategically valuable. |
| 8 | REVIEW | 130,932 impressions with position 40.1 produces a high priority from exposure combined with poor position. | Medium. | A poor average position alone does not prove that the content is the cause of the ranking outcome. |
| 9 | REVIEW | 233,561 impressions with position 26.2 indicates meaningful exposure without page-1 visibility. | Medium. | The page may already be well matched to its intended audience, making a ranking intervention low value. |
| 10 | REVIEW | 205,915 impressions with position 27.9 creates a substantial exposure-position score. | Medium. | The aggregate 90-day metrics may hide query-level differences that make the recommended review less useful. |

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = queue.head(10).copy()

display(
    top10[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ]
)


,rank,content_id,score,reason_code,action,impressions_90d,avg_position,ctr
0,1,content_a023517539fe,16224762.6,visible_but_page2plus,REVIEW,214047,85.8,0.01
1,2,content_2dba2b1f9536,7937468.6,visible_but_page2plus,REVIEW,443434,27.9,0.21
2,3,content_2cb567c3c89b,6072269.4,visible_but_page2plus,REVIEW,497727,22.2,0.10
3,4,content_54baba704595,4832829.0,visible_but_page2plus,REVIEW,130617,47.0,0.01
4,5,content_b28d1efd668f,4643049.6,visible_but_page2plus,REVIEW,286608,26.2,0.06
5,6,content_109f8f7c9d39,4017134.4,visible_but_page2plus,REVIEW,90476,54.4,0.01
6,7,content_ff94c9b6b411,3977048.4,visible_but_page2plus,REVIEW,228566,27.4,0.04
7,8,content_88d367c507a3,3941053.2,visible_but_page2plus,REVIEW,130932,40.1,0.04
8,9,content_813e88069237,3783688.2,visible_but_page2plus,REVIEW,233561,26.2,0.06
9,10,content_b511d4bc4ad2,3685878.5,visible_but_page2plus,REVIEW,205915,27.9,0.14


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak pick

**Rank 3 is a potential weak pick.** It receives a very high score mainly because of its extremely large impression volume (497,727), even though its average position is 22.2. This suggests that the rule may overweight exposure relative to the size of the position gap. The ranking is useful as decision-support, but it may prioritize very high-volume pages whose opportunity is not necessarily actionable.

---

### Precision@10 interpretation

Precision@10 was 0.3000 versus a base rate of 0.5421 on the evaluated slice. The baseline therefore did not outperform the base rate for the declining label.

This result is directional and should not be interpreted as evidence that the rule is useless. The rule was designed as a simple exposure-and-position review heuristic, while the evaluation label represents recent impression decline. The mismatch suggests that exposure plus position alone is not sufficient to identify declining content reliably.

---

### Leakage check

The baseline rule uses only `impressions_90d` and `avg_position`. It does not use `trend_pct`, `trend_direction`, or `is_declining_label`. No future-window or target-derived input is used in the score.

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Explicit leakage-feature audit
forbidden_features = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

rule_features = [
    "impressions_90d",
    "avg_position"
]

print("Rule features:")
for feature in rule_features:
    print(f"  {feature}")

print("\nForbidden leakage features:")
for feature in forbidden_features:
    print(f"  {feature}: {'PRESENT' if feature in rule_features else 'NOT USED'}")

assert not any(feature in rule_features for feature in forbidden_features)

print("\nLeakage feature check: PASS")

#proof

output_path = "work/outputs/baseline_action_score.csv"

print("CSV exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", os.path.getsize(output_path), "bytes")


# Evaluation only:
# trend_direction is the source of the target label,
# but it is NOT used anywhere in the baseline rule.

labels_for_queue = (
    df.set_index("content_id")
      .loc[queue["content_id"], "trend_direction"]
      .eq("down")
      .astype(int)
      .to_numpy()
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 10

precision10 = precision_at_k(
    queue["score"].to_numpy(),
    labels_for_queue,
    k
)

base_rate = labels_for_queue.mean()

print(f"Precision@{k}: {precision10:.4f}")
print(f"Base rate: {base_rate:.4f}")

print("Baseline exceeds base rate:",
      precision10 > base_rate)


print("Baseline rule features:", rule_features)

print("\nAvailable target/source columns:")
print([
    c for c in ["trend_direction", "trend_pct", "is_declining_label"]
    if c in df.columns
])

print("\nOutput exists:", os.path.exists("work/outputs/baseline_action_score.csv"))
print("Top-10 rows:", len(queue.head(10)))

rule_features = ["impressions_90d", "avg_position"]

forbidden_features = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

print("Rule features:")
for feature in rule_features:
    print(f"  - {feature}")

print("\nForbidden features:")
for feature in forbidden_features:
    print(f"  - {feature}: {'USED' if feature in rule_features else 'NOT USED'}")

assert not any(feature in rule_features for feature in forbidden_features)

print("\nLeakage feature check: PASS")


Rule features:
  impressions_90d
  avg_position

Forbidden leakage features:
  is_declining_label: NOT USED
  trend_direction: NOT USED
  trend_pct: NOT USED

Leakage feature check: PASS
CSV exists: True
File size: 1978010 bytes
Precision@10: 0.3000
Base rate: 0.5421
Baseline exceeds base rate: False
Baseline rule features: ['impressions_90d', 'avg_position']

Available target/source columns:
['trend_direction', 'trend_pct']

Output exists: True
Top-10 rows: 10
Rule features:
  - impressions_90d
  - avg_position

Forbidden features:
  - trend_pct: NOT USED
  - trend_direction: NOT USED
  - is_declining_label: NOT USED

Leakage feature check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.